# ⚖️ Tahap 4: Case Solution Reuse

**Tujuan:** Gunakan putusan lama (top-k hasil `retrieve()`) sebagai dasar untuk memprediksi solusi/amar kasus baru.

**Input:** fungsi `retrieve()` + `df` (Tahap 3)
**Output:**
- `04_predict.py` (script) / cell notebook tahap reuse
- Fungsi `predict_outcome(query)` teruji
- `SUBCPMK-3/results/predictions.csv`  (≡ `/data/results/predictions.csv`)

---
| Langkah | Isi |
|---------|-----|
| **i**   | Ekstrak solusi — dari top-k ambil `amar_putusan` → struktur `{case_id: solusi}` |
| **ii**  | Algoritma prediksi — **majority vote** & **weighted similarity** (bobot = skor) |
| **iii** | Fungsi `predict_outcome(query)` |
| **iv**  | Demo manual — 5 kasus baru → `predict_outcome()` → bandingkan dgn putusan asli |
| **v**   | Output — script `04_predict.py`, fungsi teruji, `predictions.csv` |


## Cell 29 — i. Ekstrak Solusi → `case_solutions = {case_id: solusi}`

In [ ]:
import re
from pathlib import Path

# Path hasil (≡ /data/results)
RESULTS_DIR     = Path("/content/drive/MyDrive/SUBCPMK-3/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_CSV = RESULTS_DIR / "predictions.csv"
PREDICT_SCRIPT  = Path("/content/drive/MyDrive/SUBCPMK-3/04_predict.py")

# ── Parser outcome dari amar_putusan ────────────────────────────
def parse_penjara_bulan(amar):
    """Total lama pidana penjara dalam BULAN; None bila tidak ada."""
    t = str(amar).lower()
    m = re.search(
        r'penjara\s+(?:selama\s+)?(\d+)\s*(?:\([^)]*\)\s*)?tahun'
        r'(?:\s*(?:dan\s+)?(\d+)\s*(?:\([^)]*\)\s*)?bulan)?', t)
    if m:
        return int(m.group(1)) * 12 + (int(m.group(2)) if m.group(2) else 0)
    m2 = re.search(r'penjara\s+(?:selama\s+)?(\d+)\s*(?:\([^)]*\)\s*)?bulan', t)
    return int(m2.group(1)) if m2 else None

def parse_denda(amar):
    m = re.search(r'denda\s+(?:sebesar\s+|sejumlah\s+)?rp[\s.]*([\d.\s]{3,})', str(amar).lower())
    if not m:
        return ''
    val = re.sub(r'\s+', '', m.group(1)).strip('.')
    return f'Rp {val}' if val else ''

def parse_verdict(amar):
    t = str(amar).strip().lower()
    if not t:                                                      return None
    if re.search(r'\bbebas\b|tidak\s+terbukti\s+secara\s+sah', t): return 'Bebas'
    if re.search(r'lepas\s+dari\s+segala\s+tuntutan', t):          return 'Lepas'
    if re.search(r'rehabilitasi', t):                              return 'Rehabilitasi'
    if re.search(r'pidana\s+penjara|penjara\s+selama', t) or parse_penjara_bulan(t) is not None:
        return 'Pidana penjara'
    return 'Lainnya'

def fmt_bulan(b):
    if b is None: return '-'
    return f'{b//12} thn {b%12} bln' if b % 12 else f'{b//12} thn'

# ── i. Bangun struktur {case_id: solusi} ────────────────────────
case_solutions = {}
for _, r in df.iterrows():
    amar = str(r.get('amar_putusan', '')).strip()   # ★ JANGAN fallback ke dakwaan
    valid = bool(amar)
    case_solutions[r['case_id']] = {
        'amar'          : amar,
        'verdict'       : parse_verdict(amar) if valid else None,
        'penjara_bulan' : parse_penjara_bulan(amar) if valid else None,
        'denda'         : parse_denda(amar) if valid else '',
        'valid_amar'    : valid,
    }

n_valid = sum(1 for v in case_solutions.values() if v['valid_amar'])
terukur = sum(1 for v in case_solutions.values() if v['penjara_bulan'] is not None)
print(f"✅ {len(case_solutions)} kasus | amar valid: {n_valid} | lama penjara terukur: {terukur}")
if n_valid < len(case_solutions):
    print(f"⚠️  {len(case_solutions)-n_valid} kasus TANPA amar valid -> DIKELUARKAN dari evaluasi prediksi")
    print("    (amar gagal diekstrak dari PDF; perbaiki Tahap 1/2 untuk data lengkap)")
print(f"📂 predictions.csv -> {PREDICTIONS_CSV}")
print("\n   Contoh:")
for cid in list(case_solutions)[:3]:
    s = case_solutions[cid]
    print(f"   {cid}: {s['verdict']} | {fmt_bulan(s['penjara_bulan'])} | {s['denda']}")

✅ 50 kasus | amar valid: 44 | lama penjara terukur: 34
⚠️  6 kasus TANPA amar valid -> DIKELUARKAN dari evaluasi prediksi
    (amar gagal diekstrak dari PDF; perbaiki Tahap 1/2 untuk data lengkap)
📂 predictions.csv -> /content/drive/MyDrive/SUBCPMK-3/results/predictions.csv

   Contoh:
   case_001: Pidana penjara | 7 thn 3 bln | 
   case_002: Rehabilitasi | - | 
   case_003: Pidana penjara | 4 thn 6 bln | 


## Cell 30 — ii. Algoritma Prediksi (majority vote & weighted similarity)

- **Majority vote** — tiap kasus top-k bersuara 1; verdict & lama-penjara terbanyak menang.
- **Weighted similarity** — bobot tiap suara = skor cosine-similarity; lama penjara = rata-rata berbobot.

In [ ]:
def aggregate(hits, strategy="weighted"):
    """hits: list dict {case_id, score} -> dict outcome terprediksi."""
    items = [(h["case_id"], float(h["score"]), case_solutions.get(h["case_id"], {})) for h in hits]
    # ★ keluarkan kasus tanpa amar valid (verdict None) agar tidak masuk voting
    items = [(c, s, sol) for c, s, sol in items
             if sol and sol.get("verdict") is not None]
    if not items:
        return None

    # (a) Verdict via voting
    vote = {}
    for c, s, sol in items:
        w = 1.0 if strategy == "majority" else max(s, 1e-6)
        vote[sol["verdict"]] = vote.get(sol["verdict"], 0.0) + w
    pred_verdict = max(vote, key=vote.get)

    # (b) Lama penjara di antara kasus berverdict sama
    same = [(s, sol["penjara_bulan"]) for c, s, sol in items
            if sol["verdict"] == pred_verdict and sol["penjara_bulan"] is not None]
    pred_bulan = None
    if same:
        if strategy == "majority":                 # modus (dibulatkan ke tahun)
            buck = {}
            for s, b in same:
                buck[round(b/12)*12] = buck.get(round(b/12)*12, 0) + 1
            pred_bulan = max(buck, key=buck.get)
        else:                                       # rata-rata berbobot skor
            wsum = sum(s for s, _ in same)
            pred_bulan = int(round(sum(s*b for s, b in same)/wsum)) if wsum else None

    # (c) Kasus rujukan = skor tertinggi dgn verdict sama
    basis = max([(c, s, sol) for c, s, sol in items if sol["verdict"] == pred_verdict],
                key=lambda x: x[1], default=items[0])
    bcid, _, bsol = basis

    return {
        "predicted_verdict"       : pred_verdict,
        "predicted_penjara_bulan" : pred_bulan,
        "predicted_denda"         : bsol.get("denda", ""),
        "basis_case_id"           : bcid,
        "solusi_text"             : bsol.get("amar", "")[:400],
        "supporting"              : [(c, round(s,4), sol["verdict"], sol["penjara_bulan"])
                                     for c, s, sol in items],
    }

print("✅ Fungsi aggregate() (majority & weighted) siap.")

✅ Fungsi aggregate() (majority & weighted) siap.


## Cell 31 — iii. Fungsi `predict_outcome(query)`

In [ ]:
def predict_outcome(query: str, k: int = 5, method: str = "tfidf",
                    strategy: str = "weighted", exclude_case_id=None,
                    detail: bool = False):
    """
    Prediksi solusi (amar) kasus baru dari putusan lama yang mirip.
    Default mengembalikan ringkasan solusi (str), sesuai spesifikasi tugas.
    """
    # 1) Ambil top-k kasus mirip (opsi exclude utk demo leave-one-out)
    raw  = retrieve(query, k=k + (1 if exclude_case_id else 0), method=method)
    hits = [h for h in raw if h["case_id"] != exclude_case_id][:k]

    # 2) Terapkan voting / weighting -> satu solusi
    agg = aggregate(hits, strategy=strategy)
    if agg is None:
        return {} if detail else "Tidak ada solusi yang dapat diprediksi."

    summary = agg["predicted_verdict"]
    if agg["predicted_penjara_bulan"] is not None:
        summary += f", pidana penjara {fmt_bulan(agg['predicted_penjara_bulan'])}"
    if agg["predicted_denda"]:
        summary += f", denda {agg['predicted_denda']}"

    if detail:
        agg.update({"summary": summary, "top_k": [h["case_id"] for h in hits],
                    "strategy": strategy})
        return agg
    return summary       # <- predicted_solution (str)

# ── Uji cepat ───────────────────────────────────────────────────
demo_q = "kepemilikan narkotika golongan i jenis sabu pasal 112 terdakwa"
print("Query:", demo_q, "\n")
for strat in ("majority", "weighted"):
    out = predict_outcome(demo_q, k=5, method="tfidf", strategy=strat, detail=True)
    print(f"── strategy = {strat} ──")
    print(f"   Prediksi : {out.get('summary')}")
    print(f"   Dasar    : {out.get('basis_case_id')} | top-k = {out.get('top_k')}")
    print()

Query: kepemilikan narkotika golongan i jenis sabu pasal 112 terdakwa 

── strategy = majority ──
   Prediksi : Pidana penjara, pidana penjara 2 thn
   Dasar    : case_040 | top-k = ['case_040', 'case_006', 'case_041', 'case_007', 'case_046']

── strategy = weighted ──
   Prediksi : Pidana penjara, pidana penjara 2 thn 11 bln
   Dasar    : case_040 | top-k = ['case_040', 'case_006', 'case_041', 'case_007', 'case_046']



## Cell 32 — iv. Demo Manual 5 Kasus + Simpan `predictions.csv`

Tiap kasus uji diperlakukan sebagai *kasus baru*: query dibangun dari **fakta/dakwaan + pasal** (TANPA amar, agar tidak bocor),
kasus itu sendiri di-*exclude* dari hasil retrieval (*leave-one-out*), lalu prediksi dibandingkan dengan amar aslinya.

In [ ]:
import pandas as pd

STRATEGY = "weighted"     # ganti ke "majority" bila perlu
METHOD   = "tfidf"        # atau "bert"
N_DEMO   = 5

# Pilih kasus yang amar-nya punya lama penjara terukur (agar bisa dibandingkan)
cand = [cid for cid, s in case_solutions.items() if s["penjara_bulan"] is not None]
cand = cand[:N_DEMO] if len(cand) >= N_DEMO else df["case_id"].head(N_DEMO).tolist()

rows = []
for qi, cid in enumerate(cand, start=1):
    row = df[df["case_id"] == cid].iloc[0]
    q = " ".join(str(row.get(c, "")) for c in
                 ["jenis_perkara", "pasal", "dakwaan_ringkas", "top_terms"])
    pred   = predict_outcome(q, k=5, method=METHOD, strategy=STRATEGY,
                             exclude_case_id=cid, detail=True)
    actual = case_solutions[cid]
    pb, ab = pred.get("predicted_penjara_bulan"), actual["penjara_bulan"]
    rows.append({
        # ── kolom WAJIB sesuai spesifikasi tugas ──
        "query_id"               : f"q{qi:02d}",
        "predicted_solution"     : pred.get("summary", ""),
        "top_5_case_ids"         : "|".join(pred.get("top_k", [])),
        # ── kolom analitik tambahan ──
        "case_id"                : cid,
        "query"                  : q[:200],
        "strategy"               : STRATEGY,
        "predicted_verdict"      : pred.get("predicted_verdict", ""),
        "predicted_penjara_bulan": pb if pb is not None else "",
        "predicted_denda"        : pred.get("predicted_denda", ""),
        "basis_case_id"          : pred.get("basis_case_id", ""),
        "actual_verdict"         : actual["verdict"],
        "actual_penjara_bulan"   : ab if ab is not None else "",
        "match_verdict"          : int(pred.get("predicted_verdict") == actual["verdict"]),
        "selisih_penjara_bulan"  : (abs(pb-ab) if (pb is not None and ab is not None) else ""),
    })

pred_df = pd.DataFrame(rows)
pred_df.to_csv(PREDICTIONS_CSV, index=False, encoding="utf-8-sig")

show = ["query_id","predicted_solution","top_5_case_ids","match_verdict"]
print(pred_df[show].to_string(index=False))
acc = pred_df["match_verdict"].mean() if len(pred_df) else 0
mae = pd.to_numeric(pred_df["selisih_penjara_bulan"], errors="coerce").dropna()
print(f"\n✅ Akurasi verdict : {acc:.2f}  ({pred_df['match_verdict'].sum()}/{len(pred_df)})")
if len(mae):
    print(f"   MAE lama penjara : {mae.mean():.1f} bulan (rata-rata selisih prediksi vs asli)")
print(f"📄 predictions.csv -> {PREDICTIONS_CSV}")

query_id                          predicted_solution                               top_5_case_ids  match_verdict
     q01  Pidana penjara, pidana penjara 5 thn 3 bln case_030|case_025|case_013|case_010|case_031              1
     q02 Pidana penjara, pidana penjara 3 thn 11 bln case_050|case_035|case_018|case_033|case_007              1
     q03  Pidana penjara, pidana penjara 5 thn 2 bln case_003|case_001|case_050|case_020|case_007              1
     q04  Pidana penjara, pidana penjara 1 thn 7 bln case_034|case_014|case_022|case_028|case_042              1
     q05        Pidana penjara, pidana penjara 6 thn case_040|case_041|case_047|case_024|case_044              1

✅ Akurasi verdict : 1.00  (5/5)
   MAE lama penjara : 40.0 bulan (rata-rata selisih prediksi vs asli)
📄 predictions.csv -> /content/drive/MyDrive/SUBCPMK-3/results/predictions.csv


## Cell 33 — v. Ekspor Script `04_predict.py`

In [ ]:
script_code = r'''#!/usr/bin/env python3
# ============================================================================
# 04_predict.py  --  Tahap 4: Case Solution Reuse
# ----------------------------------------------------------------------------
# Memakai putusan lama (top-k hasil retrieve) sebagai dasar untuk memprediksi
# solusi/amar kasus baru, lewat majority-vote atau weighted-similarity.
# Mesin retrieval (TF-IDF / IndoBERT) diimpor dari 03_retrieval.py.
#
# Contoh:
#   python 04_predict.py --cases processed/cases.csv \
#       --query "kepemilikan sabu golongan i 5 gram pasal 112" --strategy weighted
#   python 04_predict.py --cases processed/cases.csv --demo \
#       --results results/predictions.csv
# ============================================================================
import argparse
import importlib.util
import re
from pathlib import Path

import numpy as np
import pandas as pd


# ── Impor Retriever dari 03_retrieval.py (nama modul diawali angka) ──────────
def load_retriever_module(script_dir):
    path = Path(script_dir) / "03_retrieval.py"
    spec = importlib.util.spec_from_file_location("retrieval_mod", str(path))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


# ── i. Ekstraksi solusi & parsing outcome dari amar_putusan ─────────────────
def parse_penjara_bulan(amar):
    # Total lama pidana penjara dalam BULAN; None bila tidak ada.
    t = str(amar).lower()
    m = re.search(
        r"penjara\s+(?:selama\s+)?(\d+)\s*(?:\([^)]*\)\s*)?tahun"
        r"(?:\s*(?:dan\s+)?(\d+)\s*(?:\([^)]*\)\s*)?bulan)?",
        t,
    )
    if m:
        tahun = int(m.group(1))
        bulan = int(m.group(2)) if m.group(2) else 0
        return tahun * 12 + bulan
    m2 = re.search(r"penjara\s+(?:selama\s+)?(\d+)\s*(?:\([^)]*\)\s*)?bulan", t)
    if m2:
        return int(m2.group(1))
    return None


def parse_denda(amar):
    m = re.search(r"denda\s+(?:sebesar\s+|sejumlah\s+)?rp[\s.]*([\d.\s]{3,})", str(amar).lower())
    if not m:
        return ""
    val = re.sub(r"\s+", "", m.group(1)).strip(".")
    return f"Rp {val}" if val else ""


def parse_verdict(amar):
    t = str(amar).strip().lower()
    if not t:
        return None
    if re.search(r"\bbebas\b|tidak\s+terbukti\s+secara\s+sah", t):
        return "Bebas"
    if re.search(r"lepas\s+dari\s+segala\s+tuntutan", t):
        return "Lepas"
    if re.search(r"rehabilitasi", t):
        return "Rehabilitasi"
    if re.search(r"pidana\s+penjara|penjara\s+selama", t) or parse_penjara_bulan(t) is not None:
        return "Pidana penjara"
    return "Lainnya"


def fmt_bulan(b):
    if b is None:
        return "-"
    return f"{b // 12} thn {b % 12} bln" if b % 12 else f"{b // 12} thn"


def build_case_solutions(df):
    # Struktur: {case_id: {amar, verdict, penjara_bulan, denda}}
    sols = {}
    for _, r in df.iterrows():
        amar = str(r.get("amar_putusan", "")).strip()   # ★ tanpa fallback ke dakwaan
        valid = bool(amar)
        sols[r["case_id"]] = {
            "amar": amar,
            "verdict": parse_verdict(amar) if valid else None,
            "penjara_bulan": parse_penjara_bulan(amar) if valid else None,
            "denda": parse_denda(amar) if valid else "",
            "valid_amar": valid,
        }
    return sols


# ── ii. Algoritma prediksi ──────────────────────────────────────────────────
def aggregate(hits, case_solutions, strategy="weighted"):
    # hits: list dict {case_id, score, ...}; -> outcome terprediksi
    items = [(h["case_id"], float(h["score"]), case_solutions.get(h["case_id"], {})) for h in hits]
    items = [(cid, s, sol) for cid, s, sol in items
             if sol and sol.get("verdict") is not None]
    if not items:
        return None

    # --- verdict via vote ---
    vote = {}
    for cid, s, sol in items:
        w = 1.0 if strategy == "majority" else max(s, 1e-6)
        vote[sol["verdict"]] = vote.get(sol["verdict"], 0.0) + w
    pred_verdict = max(vote, key=vote.get)

    # --- lama penjara (numerik) di antara kasus berverdict sama ---
    same = [(s, sol["penjara_bulan"]) for cid, s, sol in items
            if sol["verdict"] == pred_verdict and sol["penjara_bulan"] is not None]
    pred_bulan = None
    if same:
        if strategy == "majority":
            # modus dibulatkan ke tahun terdekat
            buckets = {}
            for s, b in same:
                key = round(b / 12) * 12
                buckets[key] = buckets.get(key, 0) + 1
            pred_bulan = max(buckets, key=buckets.get)
        else:
            wsum = sum(s for s, _ in same)
            pred_bulan = int(round(sum(s * b for s, b in same) / wsum)) if wsum else None

    # --- kasus rujukan (basis): skor tertinggi dgn verdict sama ---
    basis = max([(cid, s, sol) for cid, s, sol in items if sol["verdict"] == pred_verdict],
                key=lambda x: x[1], default=items[0])
    basis_cid, _, basis_sol = basis

    # --- denda: ambil dari kasus rujukan ---
    pred_denda = basis_sol.get("denda", "")

    return {
        "predicted_verdict": pred_verdict,
        "predicted_penjara_bulan": pred_bulan,
        "predicted_denda": pred_denda,
        "basis_case_id": basis_cid,
        "solusi_text": basis_sol.get("amar", "")[:400],
        "supporting": [(cid, round(s, 4), sol["verdict"], sol["penjara_bulan"]) for cid, s, sol in items],
    }


# ── iii. Fungsi predict_outcome ─────────────────────────────────────────────
def make_predict_outcome(retriever, case_solutions):
    def predict_outcome(query, k=5, method="tfidf", strategy="weighted",
                        exclude_case_id=None, detail=False):
        raw = retriever.retrieve(query, k=k + (1 if exclude_case_id else 0), method=method)
        hits = [h for h in raw if h["case_id"] != exclude_case_id][:k]
        agg = aggregate(hits, case_solutions, strategy=strategy)
        if agg is None:
            return {} if detail else "Tidak ada solusi yang dapat diprediksi."
        summary = agg["predicted_verdict"]
        if agg["predicted_penjara_bulan"] is not None:
            summary += f", pidana penjara {fmt_bulan(agg['predicted_penjara_bulan'])}"
        if agg["predicted_denda"]:
            summary += f", denda {agg['predicted_denda']}"
        if detail:
            agg["summary"] = summary
            agg["top_k"] = [h["case_id"] for h in hits]
            agg["strategy"] = strategy
            return agg
        return summary
    return predict_outcome


# ── iv. Demo manual (leave-one-out) + predictions.csv ───────────────────────
def run_demo(df, predict_outcome, case_solutions, n=5, strategy="weighted", method="tfidf"):
    # Pilih kasus yang amar-nya punya lama penjara terukur supaya bisa dibandingkan
    cand = [r["case_id"] for _, r in df.iterrows()
            if case_solutions.get(r["case_id"], {}).get("penjara_bulan") is not None]
    cand = cand[:n] if len(cand) >= n else [r["case_id"] for _, r in df.head(n).iterrows()]

    rows = []
    for qi, cid in enumerate(cand, start=1):
        row = df[df["case_id"] == cid].iloc[0]
        # Query dari fakta/dakwaan + pasal + kata kunci (TANPA amar -> hindari kebocoran)
        q = " ".join(str(row.get(c, "")) for c in ["jenis_perkara", "pasal", "dakwaan_ringkas", "top_terms"])
        pred = predict_outcome(q, k=5, method=method, strategy=strategy,
                               exclude_case_id=cid, detail=True)
        actual = case_solutions[cid]
        pb, ab = pred.get("predicted_penjara_bulan"), actual["penjara_bulan"]
        rows.append({
            # --- kolom WAJIB sesuai spesifikasi tugas ---
            "query_id"          : f"q{qi:02d}",
            "predicted_solution": pred.get("summary", ""),
            "top_5_case_ids"    : "|".join(pred.get("top_k", [])),
            # --- kolom analitik tambahan ---
            "case_id": cid,
            "query": q[:200],
            "strategy": strategy,
            "predicted_verdict": pred.get("predicted_verdict", ""),
            "predicted_penjara_bulan": pb if pb is not None else "",
            "predicted_denda": pred.get("predicted_denda", ""),
            "basis_case_id": pred.get("basis_case_id", ""),
            "actual_verdict": actual["verdict"],
            "actual_penjara_bulan": ab if ab is not None else "",
            "match_verdict": int(pred.get("predicted_verdict") == actual["verdict"]),
            "selisih_penjara_bulan": (abs(pb - ab) if (pb is not None and ab is not None) else ""),
        })
    return pd.DataFrame(rows)


def main():
    ap = argparse.ArgumentParser(description="Tahap 4 Case Solution Reuse")
    ap.add_argument("--cases", required=True)
    ap.add_argument("--query", help="prediksi satu query")
    ap.add_argument("--demo", action="store_true", help="jalankan demo 5 kasus")
    ap.add_argument("--results", default="predictions.csv", help="path output predictions.csv")
    ap.add_argument("--strategy", choices=["majority", "weighted"], default="weighted")
    ap.add_argument("--method", choices=["tfidf", "bert"], default="tfidf")
    ap.add_argument("--bert", action="store_true")
    ap.add_argument("-k", type=int, default=5)
    args = ap.parse_args()

    mod = load_retriever_module(Path(__file__).resolve().parent)
    df = pd.read_csv(args.cases, encoding="utf-8-sig", dtype=str).fillna("")
    retriever = mod.Retriever(df, use_bert=(args.bert or args.method == "bert"))
    case_solutions = build_case_solutions(df)
    predict_outcome = make_predict_outcome(retriever, case_solutions)
    print(f"cases.csv: {len(df)} kasus | solusi terekstrak: {len(case_solutions)}")

    if args.query:
        out = predict_outcome(args.query, k=args.k, method=args.method,
                              strategy=args.strategy, detail=True)
        print(f"\nQuery: {args.query}")
        print(f"Prediksi: {out.get('summary')}")
        print(f"Dasar   : {out.get('basis_case_id')} | top-k: {out.get('top_k')}")

    if args.demo:
        res = run_demo(df, predict_outcome, case_solutions, n=5,
                       strategy=args.strategy, method=args.method)
        Path(args.results).parent.mkdir(parents=True, exist_ok=True)
        res.to_csv(args.results, index=False, encoding="utf-8-sig")
        print(f"\n== Demo {len(res)} kasus (strategy={args.strategy}) ==")
        print(res[["case_id", "predicted_verdict", "predicted_penjara_bulan",
                   "actual_verdict", "actual_penjara_bulan", "match_verdict"]].to_string(index=False))
        acc = res["match_verdict"].mean() if len(res) else 0
        print(f"\nAkurasi verdict: {acc:.2f} | predictions.csv -> {args.results}")


if __name__ == "__main__":
    main()
'''

PREDICT_SCRIPT.write_text(script_code, encoding="utf-8")
print(f"✅ Script tersimpan -> {PREDICT_SCRIPT}")
print("   (mengimpor Retriever dari 03_retrieval.py di folder yang sama)")
print("   Jalankan lokal: python 04_predict.py --cases processed/cases.csv \\")
print("                    --demo --results results/predictions.csv --strategy weighted")

✅ Script tersimpan -> /content/drive/MyDrive/SUBCPMK-3/04_predict.py
   (mengimpor Retriever dari 03_retrieval.py di folder yang sama)
   Jalankan lokal: python 04_predict.py --cases processed/cases.csv \
                    --demo --results results/predictions.csv --strategy weighted


---
## Catatan Tahap 4

| Komponen | Keterangan |
|----------|-----------|
| **`case_solutions`** | `{case_id: {amar, verdict, penjara_bulan, denda}}` dari `amar_putusan` |
| **Majority vote** | Suara 1 per kasus; verdict & lama-penjara terbanyak menang |
| **Weighted similarity** | Bobot suara = skor cosine; lama penjara = rata-rata berbobot |
| **`predict_outcome(query)`** | retrieve → vote/weight → ringkasan solusi (str); `detail=True` utk dict |
| **Demo** | 5 kasus *leave-one-out* (query dari fakta, amar di-exclude) |
| **`predictions.csv`** | kolom WAJIB: `query_id, predicted_solution, top_5_case_ids` (+ kolom analitik tambahan) |

**Kolom `predictions.csv`:** wajib `query_id, predicted_solution, top_5_case_ids` (sesuai spesifikasi tugas); ditambah kolom analitik `case_id, query, strategy, predicted_verdict, predicted_penjara_bulan, predicted_denda, basis_case_id, actual_verdict, actual_penjara_bulan, match_verdict, selisih_penjara_bulan`.

> Catatan akademik: prediksi lama pidana bersifat **indikatif** karena dataset kecil dan amar putusan sangat bervariasi.
> Untuk hasil lebih baik: perbanyak kasus per kategori pasal, atau jadikan lama-penjara sebagai regresi berbobot similarity (sudah diterapkan pada strategi `weighted`).
